In [1]:
# Cell 0 — 基本環境 & 版本確認（含 OpenMP 快修）
# 建議先跑（避免 Windows/Conda 的 OpenMP 衝突）
import os, sys, platform, shutil

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["ULTRALYTICS_HUB_MODEL"] = "False"
os.environ["YOLO_IGNORE_VERSION_CHECK"] = "1"


print("Python:", sys.version)
try:
    import torch, cv2, numpy as np
    print("torch:", torch.__version__, "| cuda available?", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("cuda device 0:", torch.cuda.get_device_name(0))
    print("opencv:", cv2.__version__)
    print("numpy:", np.__version__)
except Exception as e:
    print("依賴尚未完整，後續 Cell 會裝到好：", e)


Python: 3.11.13 | packaged by Anaconda, Inc. | (main, Jun  5 2025, 13:03:15) [MSC v.1929 64 bit (AMD64)]
torch: 2.10.0.dev20250923+cu128 | cuda available? True
cuda device 0: NVIDIA GeForce RTX 5070
opencv: 4.12.0
numpy: 2.2.6


In [2]:
# Cell 1 — 專案路徑設定（請依你的磁碟/資料夾調整根目錄）
from pathlib import Path

# 專案根目錄（請改成你目前 .ipynb 所在的專案根）
PROJ = Path(r"D:/中興大學/碩一上/仁愛醫院/仁愛醫院調整8總整理").resolve()

# 原始 Labelme JSON 四個 Stage 的搜尋起點（通常放在專案上一層或同層）
SEARCH_BASE = PROJ.parent  # 如果你的 Ficat stage 1~4 在專案內，就改成 PROJ

# YOLO 資料與輸出
YOLO_ROOT   = PROJ / "yolo_dataset_process"  # 改儲存位置
YOLO_DS     = YOLO_ROOT / "yolo_dataset"         # images/labels（由JSON轉出的全集）
YOLO_SPLIT  = YOLO_ROOT / "yolo_dataset_split"   # images/{train,val,test}
WEIGHTS_DIR = PROJ / "weights"
WEIGHTS_DIR.mkdir(exist_ok=True, parents=True)

for p in [YOLO_DS/ "images", YOLO_DS/"labels", YOLO_SPLIT, WEIGHTS_DIR]:
    p.mkdir(exist_ok=True, parents=True)

print("PROJ     :", PROJ)
print("SEARCH   :", SEARCH_BASE)
print("YOLO_DS  :", YOLO_DS)
print("SPLIT    :", YOLO_SPLIT)
print("WEIGHTS  :", WEIGHTS_DIR)


PROJ     : D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理
SEARCH   : D:\中興大學\碩一上\仁愛醫院
YOLO_DS  : D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\yolo_dataset_process\yolo_dataset
SPLIT    : D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\yolo_dataset_process\yolo_dataset_split
WEIGHTS  : D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\weights


In [3]:
from pathlib import Path
import json, os, re, shutil
from PIL import Image
from collections import defaultdict

# 四個 stage 的來源路徑（這個你剛剛測過是對的）
IMG_ROOTS = [
    PROJ / "drive-download-20251023T113302Z-1-001" / "Ficat stage 1",
    PROJ / "drive-download-20251023T113302Z-1-001" / "Ficat stage 2",
    PROJ / "drive-download-20251023T113302Z-1-001" / "Ficat stage 3",
    PROJ / "drive-download-20251023T113302Z-1-001" / "Ficat stage 4",
]

# 用 Cell 1 設好的 YOLO_DS
OUT_IMG = YOLO_DS / "images"
OUT_LBL = YOLO_DS / "labels"
OUT_IMG.mkdir(parents=True, exist_ok=True)
OUT_LBL.mkdir(parents=True, exist_ok=True)
print("OUT_IMG:", OUT_IMG)
print("OUT_LBL:", OUT_LBL)
OUT_IMG.mkdir(parents=True, exist_ok=True)
OUT_LBL.mkdir(parents=True, exist_ok=True)

def to_yolo_line(x1,y1,x2,y2,w,h,cls=0):
    xc = ((x1+x2)/2)/w; yc = ((y1+y2)/2)/h
    bw = abs(x2-x1)/w;  bh = abs(y2-y1)/h
    return f"{cls} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}"

def bbox_from_points(pts):
    xs = [p[0] for p in pts]; ys = [p[1] for p in pts]
    return min(xs), min(ys), max(xs), max(ys)

# key = (stage, 檔名)
agg = defaultdict(lambda: {"boxes": [], "w": None, "h": None, "src": None})
total_json = 0
missing_img = 0

for root in IMG_ROOTS:
    print("📁 掃描 stage root:", root)
    # ❗ 只用 *.json，不再混進 *.JSON，避免雙倍計算
    json_files = sorted(root.rglob("*.json"))
    print("  → 找到 json 檔：", len(json_files))
    for js in json_files:
        total_json += 1
        data = json.loads(js.read_text(encoding="utf-8"))

        m = re.search(r"Ficat\s*stage\s*(\d)", js.as_posix(), re.I)
        stage = m.group(1) if m else "X"

        img_name = data.get("imagePath", js.with_suffix(".jpg").name)
        base_name = Path(img_name).name
        key = (stage, base_name)

        # 找對應影像
        candidates = []
        for ext in [".jpg",".png",".jpeg",".JPG",".PNG"]:
            p = js.with_suffix(ext)
            if p.exists():
                candidates.append(p)
        p2 = js.parent / base_name
        if p2.exists():
            candidates.append(p2)

        if not candidates:
            missing_img += 1
        else:
            src_img = candidates[0]
            agg[key]["src"] = agg[key]["src"] or src_img

        w, h = data.get("imageWidth"), data.get("imageHeight")
        if isinstance(w,(int,float)) and isinstance(h,(int,float)):
            agg[key]["w"] = agg[key]["w"] or w
            agg[key]["h"] = agg[key]["h"] or h

        for sh in data.get("shapes", []):
            if sh.get("shape_type") not in ("rectangle","polygon"):
                continue
            x1,y1,x2,y2 = bbox_from_points(sh["points"])
            agg[key]["boxes"].append((x1,y1,x2,y2))

# === 實際輸出 ===
n_new_copied = 0
n_total_boxes = 0
n_noimg_after_all = 0

for (stage, base_name), info in sorted(agg.items()):
    new_name = f"S{stage}_{base_name}"
    dst_img = OUT_IMG / new_name
    src_img = info["src"]

    if src_img is None or not src_img.exists():
        n_noimg_after_all += 1
        continue

    if not (isinstance(info["w"], (int,float)) and isinstance(info["h"], (int,float))):
        with Image.open(src_img) as im:
            info["w"], info["h"] = im.size

    if not dst_img.exists():
        try:
            os.link(src_img, dst_img)
        except Exception:
            shutil.copy2(src_img, dst_img)
        n_new_copied += 1

    lines = [to_yolo_line(x1,y1,x2,y2,info["w"],info["h"],cls=0) for (x1,y1,x2,y2) in info["boxes"]]
    (OUT_LBL / (Path(new_name).with_suffix(".txt").name)).write_text("\n".join(lines), encoding="utf-8")
    n_total_boxes += len(lines)

all_imgs_now = [p for p in OUT_IMG.glob("*") if p.is_file()]

print("\n🧾 最終統計：")
print(f"  JSON 檔數              ：{total_json}")
print(f"  唯一 (stage, 檔名) key：{len(agg)}  ← 理論張數")
print(f"  這一輪新複製影像數    ：{n_new_copied}")
print(f"  目前資料夾內實際影像數：{len(all_imgs_now)}")
print(f"  總框數                ：{n_total_boxes}")
print(f"  找不到影像的 JSON     ：{missing_img}")
print(f"  最終仍無法定位影像 key：{n_noimg_after_all}")
print(f"✅ YOLO 影像目錄: {OUT_IMG}")
print(f"✅ YOLO 標註目錄: {OUT_LBL}")


OUT_IMG: D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\yolo_dataset_process\yolo_dataset\images
OUT_LBL: D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\yolo_dataset_process\yolo_dataset\labels
📁 掃描 stage root: D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\drive-download-20251023T113302Z-1-001\Ficat stage 1
  → 找到 json 檔： 49
📁 掃描 stage root: D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\drive-download-20251023T113302Z-1-001\Ficat stage 2
  → 找到 json 檔： 112
📁 掃描 stage root: D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\drive-download-20251023T113302Z-1-001\Ficat stage 3
  → 找到 json 檔： 60
📁 掃描 stage root: D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\drive-download-20251023T113302Z-1-001\Ficat stage 4
  → 找到 json 檔： 66

🧾 最終統計：
  JSON 檔數              ：287
  唯一 (stage, 檔名) key：287  ← 理論張數
  這一輪新複製影像數    ：287
  目前資料夾內實際影像數：287
  總框數                ：441
  找不到影像的 JSON     ：0
  最終仍無法定位影像 key：0
✅ YOLO 影像目錄: D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\yolo_dataset_process\yolo_dataset\images
✅ YOLO 標註目錄: D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\yolo_dataset_process\yolo_dataset\labels


In [4]:
# Cell 3 — 切分 train/val/test（不看病人ID；想嚴謹可改 ID 版）
import random
from pathlib import Path
random.seed(42)

img_dir = YOLO_DS/"images"
lbl_dir = YOLO_DS/"labels"

all_imgs = sorted([p for p in img_dir.glob("*.*") if p.suffix.lower() in [".jpg",".png",".jpeg",".bmp",".tif",".tiff"]])
print("找到影像張數：", len(all_imgs))

# 7/2/1
random.shuffle(all_imgs)
n = len(all_imgs)
n_train, n_val = int(0.7*n), int(0.9*n)
splits = {
    "train": all_imgs[:n_train],
    "val":   all_imgs[n_train:n_val],
    "test":  all_imgs[n_val:]
}

def copy_split(split_name, files):
    dst_img = YOLO_SPLIT / "images" / split_name
    dst_lbl = YOLO_SPLIT / "labels" / split_name
    dst_img.mkdir(parents=True, exist_ok=True)
    dst_lbl.mkdir(parents=True, exist_ok=True)
    has, miss = 0,0
    for img in files:
        # 複製影像
        di = dst_img / img.name
        if not di.exists():
            try: os.link(img, di)
            except Exception: shutil.copy2(img, di)
        # 複製對應 label（沒有就空 txt）
        lt = lbl_dir / (img.stem + ".txt")
        dl = dst_lbl / (img.stem + ".txt")
        if lt.exists():
            dl.write_text(lt.read_text(encoding="utf-8"), encoding="utf-8")
            has += 1
        else:
            dl.write_text("", encoding="utf-8")
            miss += 1
    print(f"{split_name}: {len(files)} imgs（有標籤 {has}，缺標籤 {miss}）")

for k, v in splits.items():
    copy_split(k, v)
print("✅ 完成隨機切分")



找到影像張數： 287
train: 200 imgs（有標籤 200，缺標籤 0）
val: 58 imgs（有標籤 58，缺標籤 0）
test: 29 imgs（有標籤 29，缺標籤 0）
✅ 完成隨機切分


In [5]:
# Cell 4 — 生成 data.yaml（用絕對路徑最穩）
yaml_text = (
    "train: " + (YOLO_SPLIT/"images"/"train").as_posix() + "\n" +
    "val:   " + (YOLO_SPLIT/"images"/"val").as_posix()   + "\n" +
    "test:  " + (YOLO_SPLIT/"images"/"test").as_posix()  + "\n" +
    "nc: 1\n"
    "names: [hip_target]\n"
)

DATA_YAML = YOLO_ROOT/"data.yaml"
DATA_YAML.write_text(yaml_text, encoding="utf-8")

print(DATA_YAML.read_text(encoding="utf-8"))
print("✅ 寫入 data.yaml >", DATA_YAML)



train: D:/中興大學/碩一上/仁愛醫院/仁愛醫院調整8總整理/yolo_dataset_process/yolo_dataset_split/images/train
val:   D:/中興大學/碩一上/仁愛醫院/仁愛醫院調整8總整理/yolo_dataset_process/yolo_dataset_split/images/val
test:  D:/中興大學/碩一上/仁愛醫院/仁愛醫院調整8總整理/yolo_dataset_process/yolo_dataset_split/images/test
nc: 1
names: [hip_target]

✅ 寫入 data.yaml > D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\yolo_dataset_process\data.yaml


In [6]:
# Cell 5 — 訓練 YOLOv8（自動選用 GPU，如不可用則退回 CPU）
# 若尚未安裝：pip install ultralytics
from ultralytics import YOLO
import torch

model = YOLO(str((WEIGHTS_DIR/"yolov8n.pt")))  # 你也可以改成 "yolov8n.pt" 自動下載
device = "0" if torch.cuda.is_available() else "cpu"
print("使用 device =", device)

train_res = model.train(
    data=str(DATA_YAML),
    epochs=100,
    imgsz=640,
    device=device,
    name="train_nb",      # run 名稱
    project=str(YOLO_ROOT/"runs"/"detect")
)


使用 device = 0
New https://pypi.org/project/ultralytics/8.3.252 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.228  Python-3.11.13 torch-2.10.0.dev20250923+cu128 CUDA:0 (NVIDIA GeForce RTX 5070, 12227MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\\\\8\yolo_dataset_process\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=D:\\\\8\weights\yolov8n.pt, momentum=0.937, mosaic=1

In [7]:
# Cell 6 — 對 test 推論（並存 .txt，之後要轉 pred.csv）
best = YOLO_ROOT/"runs"/"detect"/"train_nb"/"weights"/"best.pt"
assert best.exists(), f"找不到 {best}, 請確認訓練是否完成"

pred_res = model.predict(
    source=str(YOLO_SPLIT/"images"/"test"),
    conf=0.01,          # 全收，讓評估自己挑最適合的
    save_txt=True,
    save_conf=True,
    device=device,
    name="predict_nb",
    project=str(YOLO_ROOT/"runs"/"detect")
)

pred_labels = YOLO_ROOT/"runs"/"detect"/"predict_nb"/"labels"
print("labels dir:", pred_labels, "| 存在？", pred_labels.exists())



image 1/29 D:\\\\8\yolo_dataset_process\yolo_dataset_split\images\test\S1_L12.jpg: 320x640 1 hip_target, 20.1ms
image 2/29 D:\\\\8\yolo_dataset_process\yolo_dataset_split\images\test\S1_L6.jpg: 320x640 1 hip_target, 5.9ms
image 3/29 D:\\\\8\yolo_dataset_process\yolo_dataset_split\images\test\S1_L7.jpg: 320x640 2 hip_targets, 4.2ms
image 4/29 D:\\\\8\yolo_dataset_process\yolo_dataset_split\images\test\S1_L9.jpg: 640x384 4 hip_targets, 20.3ms
image 5/29 D:\\\\8\yolo_dataset_process\yolo_dataset_split\images\test\S1_R1.jpg: 320x640 2 hip_targets, 4.7ms
image 6/29 D:\\\\8\yolo_dataset_process\yolo_dataset_split\images\test\S1_R5.jpg: 320x640 1 hip_target, 4.7ms
image 7/29 D:\\\\8\yolo_dataset_process\yolo_dataset_split\images\test\S1_R8.jpg: 320x640 1 hip_target, 5.6ms
image 8/29 D:\\\\8\yolo_dataset_process\yolo_dataset_split\images\test\S2_L12.jpg: 320x640 1 hip_target, 5.5ms
image 9/29 D:\\\\8\yolo_dataset_process\yolo_dataset_split\images\test\S2_L17.jpg: 320x640 1 hip_target, 5.0ms
i

In [8]:
# Cell 7 — YOLO .txt（相對座標）→ pred.csv（絕對 xyxy）
import pandas as pd
from PIL import Image

LABELS_DIR   = pred_labels
TEST_IMG_DIR = YOLO_SPLIT/"images"/"test"

rows=[]
for txt in sorted(LABELS_DIR.glob("*.txt")):
    stem = txt.stem
    # 找原圖
    img = None
    for ext in (".jpg",".png",".jpeg",".bmp",".tif",".tiff",".JPG",".PNG"):
        p = TEST_IMG_DIR / f"{stem}{ext}"
        if p.exists(): img = p; break
    if img is None:
        cands = list(TEST_IMG_DIR.glob(stem + ".*"))
        if not cands:
            print("⚠️ 找不到對應影像：", stem); 
            continue
        img = cands[0]
    with Image.open(img) as im:
        w,h = im.size

    # YOLO txt: class cx cy w h conf（相對座標）
    for line in txt.read_text().strip().splitlines():
        parts = line.split()
        if len(parts) < 6: 
            continue
        _, cx, cy, bw, bh, conf = map(float, parts)
        abs_w, abs_h = bw*w, bh*h
        cx, cy = cx*w, cy*h
        x1, y1 = cx - abs_w/2, cy - abs_h/2
        x2, y2 = cx + abs_w/2, cy + abs_h/2
        rows.append({"filename": img.name, "x1": x1, "y1": y1, "x2": x2, "y2": y2, "score": conf})

pred_csv = PROJ/"pred.csv"
pd.DataFrame(rows).to_csv(pred_csv, index=False, encoding="utf-8-sig")
print(f"✅ 產生 pred.csv：{pred_csv}（{len(rows)} 筆）")


✅ 產生 pred.csv：D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\pred.csv（54 筆）


In [9]:
# 🔄 Cell 8 — 正確版：從「那套乾淨的 IMG_ROOTS」產生 gt_all.csv
import pandas as pd
import json, re
from pathlib import Path
from PIL import Image

# 這裡直接沿用你前面定義過的 IMG_ROOTS, bbox_from_points, PROJ

GT_ALL = PROJ / "gt_all.csv"

def build_gt_all_from_img_roots(img_roots, out_csv: Path):
    rows = []

    for root in img_roots:
        print("掃描 GT 來源資料夾:", root)
        # 從路徑抓出 stage 編號
        m = re.search(r"Ficat\s*stage\s*(\d)", root.as_posix(), re.I)
        if not m:
            print("  ⚠️ 無法從路徑判斷 stage，略過:", root)
            continue
        stage = int(m.group(1))

        for js in sorted(root.rglob("*.json")):   # 只吃小寫 .json
            data = json.loads(js.read_text(encoding="utf-8"))

            # 原始檔名（沒有 S 前綴）
            fname = data.get("imagePath", js.with_suffix(".jpg").name)
            base  = Path(fname).name  # L1.jpg 這種
            side  = 'L' if base.upper().startswith('L') else 'R'

            # 找原圖路徑（跟前面 YOLO 轉換邏輯一樣）
            img_path = None
            for ext in [".jpg",".png",".jpeg",".JPG",".PNG"]:
                p = js.with_suffix(ext)
                if p.exists():
                    img_path = p
                    break
            if img_path is None:
                # 找不到就略過
                continue

            # 圖片尺寸：優先用 JSON，沒有就讀檔
            w = data.get("imageWidth")
            h = data.get("imageHeight")
            if not (isinstance(w,(int,float)) and isinstance(h,(int,float))):
                with Image.open(img_path) as im:
                    w, h = im.size

            # 逐框寫入
            for sh in data.get("shapes", []):
                if sh.get("shape_type") not in ("rectangle", "polygon"):
                    continue
                x1, y1, x2, y2 = bbox_from_points(sh["points"])

                rows.append({
                    # 🔑 用跟 YOLO 一樣的命名，後面 evaluate 才對得上
                    "filename": f"S{stage}_{base}",
                    "x1": x1, "y1": y1, "x2": x2, "y2": y2,
                    "side": side,
                    "stage": stage,
                    "ID": Path(base).stem,
                    "img_w": w,
                    "img_h": h,
                })

    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False, encoding="utf-8-sig")
    print("\n✅ 產生 GT CSV：", out_csv)
    print("  總框數：", len(df))
    print("  依 stage 統計：")
    print(df["stage"].value_counts().sort_index())
    return df


# 先刪掉剛剛錯誤版本（如果存在）
if GT_ALL.exists():
    print("刪除舊的 gt_all.csv：", GT_ALL)
    GT_ALL.unlink()

df_gt = build_gt_all_from_img_roots(IMG_ROOTS, GT_ALL)
df_gt.head()




掃描 GT 來源資料夾: D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\drive-download-20251023T113302Z-1-001\Ficat stage 1
掃描 GT 來源資料夾: D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\drive-download-20251023T113302Z-1-001\Ficat stage 2
掃描 GT 來源資料夾: D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\drive-download-20251023T113302Z-1-001\Ficat stage 3
掃描 GT 來源資料夾: D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\drive-download-20251023T113302Z-1-001\Ficat stage 4

✅ 產生 GT CSV： D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\gt_all.csv
  總框數： 441
  依 stage 統計：
stage
1     66
2    159
3     95
4    121
Name: count, dtype: int64


,filename,x1,y1,x2,y2,side,stage,ID,img_w,img_h
0,S1_L1.jpg,626.574074,219.135802,898.179012,531.481481,L,1,L1,1905,900
1,S1_L1.jpg,1077.191358,229.012346,1348.796296,557.407407,L,1,L1,1905,900
2,S1_L10.jpg,25.038835,210.815534,302.708738,594.310680,L,1,L10,786,876
3,S1_L10.jpg,465.815534,232.174757,751.252427,595.281553,L,1,L10,786,876
4,S1_L11.jpg,512.993827,185.802469,846.327160,582.098765,L,1,L11,1905,900


In [10]:
# Cell 9 — 只取 test 名單 的 GT → gt_test.csv
df_gt = pd.read_csv(GT_ALL)
test_files = sorted([p.name for p in (YOLO_SPLIT/"images"/"test").glob("*.*")])
df_gt_test = df_gt[df_gt["filename"].isin(test_files)].copy()

GT_TEST = PROJ/"gt_test.csv"
df_gt_test.to_csv(GT_TEST, index=False, encoding="utf-8-sig")
print("GT(all) =", len(df_gt), "→ GT(test) =", len(df_gt_test))
print("✅ 輸出 gt_test.csv >", GT_TEST)


GT(all) = 441 → GT(test) = 42
✅ 輸出 gt_test.csv > D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\gt_test.csv


In [11]:
# Cell 10 — 評估
import pandas as pd

gt = pd.read_csv(GT_TEST)
pred = pd.read_csv(pred_csv)

detail = []   # 最終明細，每筆=一個GT框

for idx, g in gt.iterrows():
    df = pred[pred["filename"] == g["filename"]]
    best_iou = 0
    for _, p in df.iterrows():
        xA = max(g.x1, p.x1)
        yA = max(g.y1, p.y1)
        xB = min(g.x2, p.x2)
        yB = min(g.y2, p.y2)
        inter = max(0, xB-xA) * max(0, yB-yA)
        area_g = (g.x2-g.x1)*(g.y2-g.y1)
        area_p = (p.x2-p.x1)*(p.y2-p.y1)
        union = area_g + area_p - inter
        iou = inter / union if union>0 else 0
        best_iou = max(best_iou, iou)

    detail.append({
        "filename": g.filename,
        "stage": g.stage,
        "IoU": best_iou
    })

detail_df = pd.DataFrame(detail)
detail_df.to_csv(PROJ/"detail_iou.csv", index=False, encoding="utf-8-sig")
detail_df.head()


,filename,stage,IoU
0,S1_L12.jpg,1,0.940086
1,S1_L6.jpg,1,0.826970
2,S1_L7.jpg,1,0.813740
3,S1_L7.jpg,1,0.761646
4,S1_L9.jpg,1,0.959766


In [12]:
# Cell 11 — 視覺化 test set：畫上「預測結果」的框

import cv2
import pandas as pd
from pathlib import Path

#=== 路徑 & 檔案 ===#
TEST_IMG_DIR = YOLO_SPLIT / "images" / "test"
PRED_CSV     = pred_csv          # 如果你上面有 pred_csv 變數，就用這個
# 如果是重新開 kernel，也可以手動指定：
# PRED_CSV = PROJ / "pred.csv"

OUT_DIR = PROJ / "vis_test_pred"  # 輸出資料夾
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("讀取 pred.csv :", PRED_CSV)
pred = pd.read_csv(PRED_CSV)

# 依檔名分組，方便逐張圖處理
grouped_pred = pred.groupby("filename")

# 可調整：只畫分數高於這個閾值的預測
CONF_TH = 0.3

n_done = 0
for img_path in sorted(TEST_IMG_DIR.glob("*.*")):
    fn = img_path.name
    img = cv2.imread(str(img_path))
    if img is None:
        print("讀圖失敗:", img_path)
        continue

    h, w = img.shape[:2]

    # 取出這張圖的所有預測框
    if fn in grouped_pred.groups:
        df_p = grouped_pred.get_group(fn)
        for _, row in df_p.iterrows():
            if row["score"] < CONF_TH:
                continue
            x1, y1, x2, y2 = map(int, [row.x1, row.y1, row.x2, row.y2])

            # 綠色框 = 預測框
            cv2.rectangle(img, (x1, y1), (x2, y2), (0,255,0), 2)

            # 在左上角寫上分數
            label = f"{row.score:.2f}"
            cv2.putText(img, label, (x1, max(0, y1-5)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1, cv2.LINE_AA)

    # 存成新圖
    out_path = OUT_DIR / fn
    cv2.imwrite(str(out_path), img)
    n_done += 1

print(f"✅ 完成視覺化預測框：{n_done} 張")
print("輸出資料夾：", OUT_DIR)


讀取 pred.csv : D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\pred.csv
✅ 完成視覺化預測框：29 張
輸出資料夾： D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\vis_test_pred
